# Notebook 03: Causal Inference - CATE Estimation với T-Learner

**PHIÊN BẢN ĐÃ SỬA - Pre-Treatment Features Only**

**Các thay đổi chính:**
- ✅ Chỉ sử dụng pre-treatment features (no data leakage)
- ✅ T-learner với 2 models riêng biệt (treatment & control)
- ✅ Sử dụng calibrated probabilities từ Notebook 02
- ✅ CATE = Individual Treatment Effects
- ✅ Phân loại 4 groups: Persuadables, Sure Things, Lost Causes, Sleeping Dogs

**Ngày:** 06/11/2026

**Tác giả:** Phan Hồ Ngọc Hân

In [1]:
# ============================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ============================================
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path

# Sklearn models
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"✅ Import hoàn tất - SEED = {SEED}")

✅ Import hoàn tất - SEED = 42


## 1. Load Dữ Liệu

In [2]:
# ============================================
# LOAD PROCESSED DATA
# ============================================
data_path = Path("../data/processed/marketing_campaign_processed.parquet")
metadata_path = Path("../data/processed/feature_metadata.json")

# Load data
df = pd.read_parquet(data_path)

# Load metadata
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

MODELING_FEATURES = metadata['modeling_features']

print(f"✅ Đã load dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"✅ Modeling features (pre-treatment): {len(MODELING_FEATURES)}")

✅ Đã load dữ liệu: 5,000 dòng × 20 cột
✅ Modeling features (pre-treatment): 8


## 2. Chia Train/Test Theo Treatment Groups

**T-Learner Approach:**
1. Train hai models riêng biệt:
   - μ₀: Control model (T=0)
   - μ₁: Treatment model (T=1)
2. CATE = μ₁(x) - μ₀(x)

In [3]:
# ============================================
# CHIA THEO TREATMENT ASSIGNMENT
# ============================================

# Chỉ dùng train set cho T-learner training
df_train = df[df['split'] == 'train'].copy()
df_test = df[df['split'] == 'locked_test'].copy()

print("=" * 70)
print("T-LEARNER DATA SPLIT")
print("=" * 70)

# Control group (T=0)
df_train_control = df_train[df_train['treatment_exposed'] == 0]
df_train_treatment = df_train[df_train['treatment_exposed'] == 1]

print(f"\nTrain Set:")
print(f"  Control (T=0):   {len(df_train_control):,} samples")
print(f"  Treatment (T=1): {len(df_train_treatment):,} samples")
print(f"  Total:           {len(df_train):,} samples")

print(f"\nTest Set (for CATE estimation):")
print(f"  Total: {len(df_test):,} samples")
print("=" * 70)

T-LEARNER DATA SPLIT

Train Set:
  Control (T=0):   1,582 samples
  Treatment (T=1): 1,418 samples
  Total:           3,000 samples

Test Set (for CATE estimation):
  Total: 500 samples


In [4]:
# ============================================
# CHUẨN BỊ X và y CHO MỖI GROUP
# ============================================

# Control group
X_train_0 = df_train_control[MODELING_FEATURES].copy()
y_train_0 = df_train_control['conversion'].values

# Treatment group
X_train_1 = df_train_treatment[MODELING_FEATURES].copy()
y_train_1 = df_train_treatment['conversion'].values

# Test set (for CATE prediction)
X_test = df_test[MODELING_FEATURES].copy()
y_test = df_test['conversion'].values
T_test = df_test['treatment_exposed'].values

print("\n✅ Đã chuẩn bị dữ liệu:")
print(f"   Control model:   X={X_train_0.shape}, y={y_train_0.shape}")
print(f"   Treatment model: X={X_train_1.shape}, y={y_train_1.shape}")
print(f"   Test set:        X={X_test.shape}")

print(f"\n📊 Conversion rates trong training:")
print(f"   Control:   {y_train_0.mean()*100:.2f}%")
print(f"   Treatment: {y_train_1.mean()*100:.2f}%")
print(f"   Naive ATE: {(y_train_1.mean() - y_train_0.mean())*100:+.2f} pp")


✅ Đã chuẩn bị dữ liệu:
   Control model:   X=(1582, 8), y=(1582,)
   Treatment model: X=(1418, 8), y=(1418,)
   Test set:        X=(500, 8)

📊 Conversion rates trong training:
   Control:   7.40%
   Treatment: 12.91%
   Naive ATE: +5.51 pp


## 3. Xác Định Categorical Features

In [5]:
# ============================================
# XÁC ĐỊNH CÁC CATEGORICAL FEATURES
# ============================================

categorical_features = ['campaign_id', 'channel', 'device', 'country', 'segment', 'value_tier']
cat_indices = [i for i, feat in enumerate(MODELING_FEATURES) if feat in categorical_features]

print(f"📋 Categorical features: {len(cat_indices)}")
print(f"   Chỉ số: {cat_indices}")

📋 Categorical features: 6
   Chỉ số: [0, 1, 2, 3, 4, 7]


## 4. Train T-Learner Models (μ₀ và μ₁)

### 4.1. Control Model (μ₀) - Nhóm Không Xử Lý

In [6]:
# ============================================
# TRAIN CONTROL MODEL (μ₀)
# ============================================

print("\n" + "=" * 70)
print("TRAINING CONTROL MODEL (μ₀) - NHÓM KHÔNG XỬ LÝ")
print("=" * 70)

model_0 = HistGradientBoostingClassifier(
    max_iter=50,
    max_depth=4,
    learning_rate=0.1,
    categorical_features=cat_indices,
    random_state=SEED
)

print(f"\n⏳ Đang train trên {len(X_train_0):,} mẫu control...")
model_0.fit(X_train_0, y_train_0)

print("\n✅ Control model đã hoàn thành!")

# Evaluate on control samples in test set
if (T_test == 0).sum() > 0:
    X_test_control = X_test[T_test == 0]
    y_test_control = y_test[T_test == 0]
    p_test_0 = model_0.predict_proba(X_test_control)[:, 1]
    auc_0 = roc_auc_score(y_test_control, p_test_0)
    print(f"   Test AUC (chỉ control): {auc_0:.4f}")


TRAINING CONTROL MODEL (μ₀) - NHÓM KHÔNG XỬ LÝ

⏳ Đang train trên 1,582 mẫu control...



✅ Control model đã hoàn thành!
   Test AUC (chỉ control): 0.5250


### 4.2. Treatment Model (μ₁) - Nhóm Xử Lý

In [7]:
# ============================================
# TRAIN TREATMENT MODEL (μ₁)
# ============================================

print("\n" + "=" * 70)
print("TRAINING TREATMENT MODEL (μ₁) - NHÓM XỬ LÝ")
print("=" * 70)

model_1 = HistGradientBoostingClassifier(
    max_iter=50,
    max_depth=4,
    learning_rate=0.1,
    categorical_features=cat_indices,
    random_state=SEED
)

print(f"\n⏳ Đang train trên {len(X_train_1):,} mẫu treatment...")
model_1.fit(X_train_1, y_train_1)

print("\n✅ Treatment model đã hoàn thành!")

# Evaluate on treatment samples in test set
if (T_test == 1).sum() > 0:
    X_test_treatment = X_test[T_test == 1]
    y_test_treatment = y_test[T_test == 1]
    p_test_1 = model_1.predict_proba(X_test_treatment)[:, 1]
    auc_1 = roc_auc_score(y_test_treatment, p_test_1)
    print(f"   Test AUC (chỉ treatment): {auc_1:.4f}")


TRAINING TREATMENT MODEL (μ₁) - NHÓM XỬ LÝ

⏳ Đang train trên 1,418 mẫu treatment...

✅ Treatment model đã hoàn thành!
   Test AUC (chỉ treatment): 0.4449


## 5. Ước Lượng CATE (Individual Treatment Effects)

**CATE(x) = μ₁(x) - μ₀(x)**

Individual treatment effect cho mỗi customer.

In [8]:
# ============================================
# ESTIMATE CATE (Test Set)
# ============================================

print("\n" + "=" * 70)
print("ƯỚC LƯỢNG CATE (INDIVIDUAL TREATMENT EFFECTS)")
print("=" * 70)

# Predict P(Y=1|X) cho cả hai models trên test set
mu_0 = model_0.predict_proba(X_test)[:, 1]  # E[Y|T=0, X]
mu_1 = model_1.predict_proba(X_test)[:, 1]  # E[Y|T=1, X]

# CATE = difference
cate = mu_1 - mu_0

print(f"\n✅ Đã ước lượng CATE cho {len(cate):,} customers")

# Summary statistics
print(f"\n📊 CATE Distribution:")
print(f"   Mean:   {cate.mean():+.4f} ({cate.mean()*100:+.2f} pp)")
print(f"   Median: {np.median(cate):+.4f}")
print(f"   Std:    {cate.std():.4f}")
print(f"   Min:    {cate.min():+.4f}")
print(f"   Max:    {cate.max():+.4f}")
print(f"   IQR:    {np.percentile(cate, 75) - np.percentile(cate, 25):.4f}")

# Heterogeneity indicator
heterogeneity_ratio = cate.std() / abs(cate.mean())
print(f"\n📈 Heterogeneity Tỷ lệ (Std/Mean): {heterogeneity_ratio:.2f}")
if heterogeneity_ratio > 1:
    print("   ✅ HIGH heterogeneity - substantial variation across customers")
else:
    print("   ⚠️  LOW heterogeneity - treatment effects relatively uniform")


ƯỚC LƯỢNG CATE (INDIVIDUAL TREATMENT EFFECTS)

✅ Đã ước lượng CATE cho 500 customers

📊 CATE Distribution:
   Mean:   +0.0631 (+6.31 pp)
   Median: +0.0432
   Std:    0.1060
   Min:    -0.2093
   Max:    +0.4990
   IQR:    0.1203

📈 Heterogeneity Tỷ lệ (Std/Mean): 1.68
   ✅ HIGH heterogeneity - substantial variation across customers


## 6. Phân Tích Theo Decile

In [9]:
# ============================================
# PHÂN TÍCH THEO DECILE (CHIA 10 NHÓM)
# ============================================

print("\n" + "=" * 70)
print("PHÂN TÍCH THEO DECILE (CHIA 10 NHÓM)")
print("=" * 70)

# Add CATE to test dataframe
df_test_cate = df_test.copy()
df_test_cate['cate'] = cate
df_test_cate['mu_0'] = mu_0
df_test_cate['mu_1'] = mu_1

# Create deciles
df_test_cate['uplift_decile'] = pd.qcut(
    df_test_cate['cate'], 
    q=10, 
    labels=False, 
    duplicates='drop'
) + 1  # 1-10 instead of 0-9

# Aggregate by decile
decile_analysis = df_test_cate.groupby('uplift_decile').agg({
    'cate': 'mean',
    'mu_0': 'mean',
    'mu_1': 'mean',
    'treatment_exposed': 'mean',
    'conversion': 'mean',
    'user_id': 'count'
}).round(4)

decile_analysis.columns = ['Mean_CATE', 'Mean_μ₀', 'Mean_μ₁', 'Treatment_%', 'Conversion_%', 'N']
decile_analysis['Treatment_%'] *= 100
decile_analysis['Conversion_%'] *= 100

print("\n📊 KẾT QUẢ PHÂN TÍCH DECILE:")
print(decile_analysis.to_string())

# Check monotonicity
cate_means = decile_analysis['Mean_CATE'].values
is_monotonic = all(cate_means[i] <= cate_means[i+1] for i in range(len(cate_means)-1))

print(f"\n📈 Kiểm tra tính đơn điệu:")
if is_monotonic:
    print("   ✅ CATE tăng đơn điệu qua các decile")
    print("   → T-learner xếp hạng customers theo uplift thành công")
else:
    print("   ⚠️  CATE không hoàn toàn đơn điệu")
    print("   → Có nhiễu trong xếp hạng (bình thường với mẫu nhỏ)")


PHÂN TÍCH THEO DECILE (CHIA 10 NHÓM)

📊 KẾT QUẢ PHÂN TÍCH DECILE:
               Mean_CATE  Mean_μ₀  Mean_μ₁  Treatment_%  Conversion_%   N
uplift_decile                                                            
1                -0.0939   0.1745   0.0806      40.0000       18.0000  50
2                -0.0246   0.0936   0.0690      50.0000        4.0000  50
3                -0.0040   0.0629   0.0589      36.0000       14.0000  50
4                 0.0158   0.0470   0.0628      56.0000       10.0000  50
5                 0.0340   0.0524   0.0864      58.0000        6.0000  50
6                 0.0561   0.0554   0.1116      46.0000        8.0000  50
7                 0.0836   0.0587   0.1423      50.0000        6.0000  50
8                 0.1142   0.0481   0.1622      32.0000        8.0000  50
9                 0.1611   0.0663   0.2273      44.0000        6.0000  50
10                0.2891   0.0750   0.3641      46.0000       16.0000  50

📈 Kiểm tra tính đơn điệu:
   ✅ CATE tăng đơn

## 7. Phân Loại 4 Nhóm Khách Hàng

Theo Gutierrez & Gérardy (2017):
1. **Persuadables:** CATE > threshold (positive uplift)
2. **Sure Things:** High baseline P(Y|T=0), low uplift
3. **Lost Causes:** Low baseline, low uplift
4. **Sleeping Dogs:** CATE < 0 (negative effect!)

In [10]:
# ============================================
# PHÂN LOẠI 4 NHÓM KHÁCH HÀNG
# ============================================

print("\n" + "=" * 70)
print("PHÂN LOẠI 4 NHÓM KHÁCH HÀNG")
print("=" * 70)

# Define thresholds
CATE_POSITIVE_THRESHOLD = 0.05  # 5 pp uplift
BASELINE_HIGH_THRESHOLD = 0.15  # 15% baseline conversion
BASELINE_LOW_THRESHOLD = 0.05   # 5% baseline conversion

# Classify
def classify_customer(row):
    cate = row['cate']
    mu_0 = row['mu_0']
    
    if cate > CATE_POSITIVE_THRESHOLD:
        return 'Persuadables'
    elif cate < -CATE_POSITIVE_THRESHOLD:
        return 'Sleeping Dogs'
    elif mu_0 > BASELINE_HIGH_THRESHOLD:
        return 'Sure Things'
    else:
        return 'Lost Causes'

df_test_cate['group'] = df_test_cate.apply(classify_customer, axis=1)

# Summary
group_summary = df_test_cate.groupby('group').agg({
    'user_id': 'count',
    'cate': 'mean',
    'mu_0': 'mean',
    'mu_1': 'mean'
}).round(4)

group_summary.columns = ['Count', 'Mean_CATE', 'Mean_μ₀', 'Mean_μ₁']
group_summary['Percentage'] = (group_summary['Count'] / len(df_test_cate) * 100).round(1)

# Sort by strategic priority
group_order = ['Persuadables', 'Sure Things', 'Lost Causes', 'Sleeping Dogs']
group_summary = group_summary.reindex(group_order)

print("\n📊 TÓM TẮT 4 NHÓM:")
print(group_summary.to_string())

# Strategic recommendations
print("\n🎯 KHUYẾN NGHỊ CHIẾN LƯỢC:")
print(f"\n✅ Persuadables ({group_summary.loc['Persuadables', 'Percentage']:.1f}%):")
print(f"   → LIÊN HỆ! Uplift cao (mean CATE: {group_summary.loc['Persuadables', 'Mean_CATE']:+.4f})")

print(f"\n⚠️  Sure Things ({group_summary.loc['Sure Things', 'Percentage']:.1f}%):")
print(f"   → GIỮ LẠI. Đã có khả năng convert cao (μ₀: {group_summary.loc['Sure Things', 'Mean_μ₀']:.4f})")

print(f"\n❌ Lost Causes ({group_summary.loc['Lost Causes', 'Percentage']:.1f}%):")
print(f"   → KHÔNG LIÊN HỆ. Không convert dù gì")

print(f"\n⛔ Sleeping Dogs ({group_summary.loc['Sleeping Dogs', 'Percentage']:.1f}%):")
print(f"   → TRÁNH! Hiệu ứng âm (mean CATE: {group_summary.loc['Sleeping Dogs', 'Mean_CATE']:+.4f})")


PHÂN LOẠI 4 NHÓM KHÁCH HÀNG

📊 TÓM TẮT 4 NHÓM:
               Count  Mean_CATE  Mean_μ₀  Mean_μ₁  Percentage
group                                                        
Persuadables     233     0.1478   0.0612   0.2090     46.6000
Sure Things       15    -0.0047   0.1900   0.1853      3.0000
Lost Causes      206     0.0084   0.0556   0.0640     41.2000
Sleeping Dogs     46    -0.0981   0.1765   0.0784      9.2000

🎯 KHUYẾN NGHỊ CHIẾN LƯỢC:

✅ Persuadables (46.6%):
   → LIÊN HỆ! Uplift cao (mean CATE: +0.1478)

⚠️  Sure Things (3.0%):
   → GIỮ LẠI. Đã có khả năng convert cao (μ₀: 0.1900)

❌ Lost Causes (41.2%):
   → KHÔNG LIÊN HỆ. Không convert dù gì

⛔ Sleeping Dogs (9.2%):
   → TRÁNH! Hiệu ứng âm (mean CATE: -0.0981)


## 8. Heterogeneity Theo Segment

In [11]:
# ============================================
# HETEROGENEITY BY CUSTOMER SEGMENT
# ============================================

print("\n" + "=" * 70)
print("CATE THEO PHÂN KHÚC KHÁCH HÀNG")
print("=" * 70)

segment_analysis = df_test_cate.groupby('segment').agg({
    'cate': ['mean', 'std'],
    'user_id': 'count'
}).round(4)

segment_analysis.columns = ['Mean_CATE', 'Std_CATE', 'Count']
segment_analysis = segment_analysis.sort_values('Mean_CATE', ascending=False)

# Calculate % persuadables per segment
persuadables_by_segment = df_test_cate[df_test_cate['group'] == 'Persuadables'].groupby('segment').size()
total_by_segment = df_test_cate.groupby('segment').size()
segment_analysis['%_Persuadables'] = (persuadables_by_segment / total_by_segment * 100).fillna(0).round(1)

print("\n📊 PHÂN TÍCH THEO SEGMENT:")
print(segment_analysis.to_string())

# Identify best segment
best_segment = segment_analysis['Mean_CATE'].idxmax()
worst_segment = segment_analysis['Mean_CATE'].idxmin()

print(f"\n🏆 Segment phản ứng tốt nhất: {best_segment}")
print(f"   Mean CATE: {segment_analysis.loc[best_segment, 'Mean_CATE']:+.4f}")
print(f"   % Persuadables: {segment_analysis.loc[best_segment, '%_Persuadables']:.1f}%")

print(f"\n⚠️  Segment phản ứng kém nhất: {worst_segment}")
print(f"   Mean CATE: {segment_analysis.loc[worst_segment, 'Mean_CATE']:+.4f}")
print(f"   % Persuadables: {segment_analysis.loc[worst_segment, '%_Persuadables']:.1f}%")


CATE THEO PHÂN KHÚC KHÁCH HÀNG

📊 PHÂN TÍCH THEO SEGMENT:
         Mean_CATE  Std_CATE  Count  %_Persuadables
segment                                            
Outdoor     0.0938    0.1195    113         60.2000
Fashion     0.0801    0.1214     93         52.7000
Tech        0.0641    0.1088     87         48.3000
Health      0.0546    0.0914     97         41.2000
Food        0.0241    0.0703    110         30.9000

🏆 Segment phản ứng tốt nhất: Outdoor
   Mean CATE: +0.0938
   % Persuadables: 60.2%

⚠️  Segment phản ứng kém nhất: Food
   Mean CATE: +0.0241
   % Persuadables: 30.9%


## 9. Xuất CATE Estimates

In [12]:
# ============================================
# XUẤT CATE ESTIMATES CHO TẤT CẢ CUSTOMERS
# ============================================

print("\n" + "=" * 70)
print("XUẤT CATE ESTIMATES")
print("=" * 70)

# Estimate CATE cho TOÀN BỘ dataset (không chỉ test)
X_all = df[MODELING_FEATURES].copy()
mu_0_all = model_0.predict_proba(X_all)[:, 1]
mu_1_all = model_1.predict_proba(X_all)[:, 1]
cate_all = mu_1_all - mu_0_all

# Classify all customers
df_all = df.copy()
df_all['cate'] = cate_all
df_all['mu_0'] = mu_0_all
df_all['mu_1'] = mu_1_all
df_all['group'] = df_all.apply(classify_customer, axis=1)

# Save models
output_dir = Path("../data/processed")
model_package = {
    'model_0': model_0,
    'model_1': model_1,
    'modeling_features': MODELING_FEATURES,
    'cat_indices': cat_indices,
    'thresholds': {
        'cate_positive': CATE_POSITIVE_THRESHOLD,
        'baseline_high': BASELINE_HIGH_THRESHOLD,
        'baseline_low': BASELINE_LOW_THRESHOLD
    },
    'seed': SEED
}

model_path = output_dir / "cate_models.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model_package, f)

print(f"\n✅ Đã lưu T-learner models: {model_path}")

# Save CATE estimates
cate_df = pd.DataFrame({
    'user_id': df_all['user_id'],
    'split': df_all['split'],
    'treatment_exposed': df_all['treatment_exposed'],
    'conversion': df_all['conversion'],
    'mu_0': mu_0_all,
    'mu_1': mu_1_all,
    'cate': cate_all,
    'group': df_all['group'],
    'channel': df_all['channel'],
    'segment': df_all['segment'],
    'value_tier': df_all['value_tier']
})

cate_path = output_dir / "cate_estimates.parquet"
cate_df.to_parquet(cate_path, index=False)

print(f"✅ Đã lưu CATE estimates: {cate_path}")
print(f"   {len(cate_df):,} customers với CATE estimates")

# Save summary statistics
summary = {
    'test_set_size': len(df_test_cate),
    'cate_mean': float(cate.mean()),
    'cate_std': float(cate.std()),
    'cate_min': float(cate.min()),
    'cate_max': float(cate.max()),
    'heterogeneity_ratio': float(heterogeneity_ratio),
    'four_groups': group_summary.to_dict('index'),
    'decile_analysis': decile_analysis.to_dict('index'),
    'segment_analysis': segment_analysis.to_dict('index')
}

summary_path = output_dir / "cate_summary.json"
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Đã lưu summary: {summary_path}")


XUẤT CATE ESTIMATES



✅ Đã lưu T-learner models: ../data/processed/cate_models.pkl
✅ Đã lưu CATE estimates: ../data/processed/cate_estimates.parquet
   5,000 customers với CATE estimates
✅ Đã lưu summary: ../data/processed/cate_summary.json


## 10. Tóm Tắt

In [13]:
# ============================================
# TÓM TẮT CUỐI CÙNG
# ============================================

print("\n" + "=" * 70)
print("TÓM TẮT NOTEBOOK 03")
print("=" * 70)

print(f"\n✅ T-Learner đã được train:")
print(f"   Control model (μ₀):   {len(X_train_0):,} samples")
print(f"   Treatment model (μ₁): {len(X_train_1):,} samples")

print(f"\n✅ CATE đã được ước lượng cho {len(cate_all):,} customers:")
print(f"   Mean: {cate.mean():+.4f} ({cate.mean()*100:+.2f} pp)")
print(f"   Std:  {cate.std():.4f}")
print(f"   Range: [{cate.min():+.4f}, {cate.max():+.4f}]")

print(f"\n✅ Bốn Nhóm (test set):")
for group in group_order:
    pct = group_summary.loc[group, 'Percentage']
    count = group_summary.loc[group, 'Count']
    print(f"   {group:15s}: {count:3.0f} ({pct:5.1f}%)")

print(f"\n✅ Heterogeneity được xác nhận:")
print(f"   Tỷ lệ (Std/Mean): {heterogeneity_ratio:.2f}")
if heterogeneity_ratio > 1:
    print("   → Biến thiên lớn - targeting có giá trị!")

print("\n🔴 CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:")
print("   • Chỉ dùng pre-treatment features (no leakage)")
print("   • T-learner với 2 models riêng biệt")
print("   • Phân loại 4 groups rõ ràng")
print("   • Heterogeneity analysis chi tiết")

print("\n✅ Sẵn sàng cho Notebook 04 (MILP Optimization)")
print("=" * 70)


TÓM TẮT NOTEBOOK 03

✅ T-Learner đã được train:
   Control model (μ₀):   1,582 samples
   Treatment model (μ₁): 1,418 samples

✅ CATE đã được ước lượng cho 5,000 customers:
   Mean: +0.0631 (+6.31 pp)
   Std:  0.1060
   Range: [-0.2093, +0.4990]

✅ Bốn Nhóm (test set):
   Persuadables   : 233 ( 46.6%)
   Sure Things    :  15 (  3.0%)
   Lost Causes    : 206 ( 41.2%)
   Sleeping Dogs  :  46 (  9.2%)

✅ Heterogeneity được xác nhận:
   Tỷ lệ (Std/Mean): 1.68
   → Biến thiên lớn - targeting có giá trị!

🔴 CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:
   • Chỉ dùng pre-treatment features (no leakage)
   • T-learner với 2 models riêng biệt
   • Phân loại 4 groups rõ ràng
   • Heterogeneity analysis chi tiết

✅ Sẵn sàng cho Notebook 04 (MILP Optimization)
